# CLIFFGUARD — round 2

**Runtime: T4 GPU.** `Runtime -> Change runtime type -> T4 GPU`, then `Run all`.

This is a **new** notebook. It does not replace `colab_run.ipynb`, and it writes
to `r2-*` run directories so nothing from the first run is overwritten.

Four gaps that need a GPU. Each answers a specific reviewer objection, and each
is independent: if the session dies after step 2, steps 1 and 2 are still worth
having.

| # | Step | Objection it answers | ~T4 |
|---|---|---|---|
| 1 | Regrade Qwen2.5-1.5B with the 7B judge | "a model was dropped for an avoidable evaluation failure" | 25 min |
| 2 | GSM8K at 496 questions | "n=200 is a toy sample, and you admit it lacks power" | 100 min |
| 3 | 256-token generations | "48 tokens structurally favours observing refusal" | 45 min |

Caches are written to Google Drive as each scheme finishes, so a disconnect
costs the scheme in flight and nothing else. Re-running the notebook resumes.
| 4 | AWQ and GPTQ checkpoints | "nobody deploys RTN; this is a sterile exercise" | 60 min, fragile |

Steps 1-3 are safe. **Step 4 is install-fragile** and is written so a failure is
recorded and skipped rather than ending the session.

## 0 — Environment

Installs only what Colab lacks. **`numpy` is deliberately not pinned** — forcing
`numpy<2` breaks Colab's preinstalled torch (ABI mismatch). `cliffguard` needs
only numpy / scipy / pydantic, all already present.


In [ ]:
import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/parnish007/CLIFFGUARD.git"
REPO_DIR = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/cliffguard")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as exc:
        print("[drive] not mounted — a disconnect will lose progress:", exc)

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "bitsandbytes", "datasets", "gguf"], check=False)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch, numpy as np, transformers

HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "NONE"
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0

print(f"repo         : {pathlib.Path.cwd()}")
print(f"python       : {platform.python_version()}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"numpy        : {np.__version__}")
print(f"GPU          : {GPU_NAME}  ({VRAM_GB} GB)")
if hasattr(os, "statvfs"):
    st = os.statvfs(".")
    print(f"free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB")

if not HAS_GPU:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.")
if tuple(int(p) for p in transformers.__version__.split(".")[:2]) < (4, 45):
    raise SystemExit(f"transformers {transformers.__version__} too old (need >= 4.45).\n"
                     "Run:  !pip -q install -U transformers   then Runtime → Restart session.")


## 1 — PREFLIGHT

Seconds, on CPU, with synthetic data — **before** any download or GPU time. It
imports every symbol the arms use, checks the signatures that matter, and
exercises each stage function.

`PREFLIGHT OK` means no arm below can die on an `ImportError`, `AttributeError`,
or wrong-arity `TypeError`. If it fails, **stop** — the notebook and the
repository have drifted, and running the arms would burn an hour producing
nothing.


In [ ]:
import numpy as np
import torch          # also imported by the setup cell; repeated so this cell stands alone

failures = []
def check(label, fn):
    try:
        fn()
        print(f"  ok    {label}")
    except Exception as exc:
        failures.append(f"{label}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {label}: {type(exc).__name__}: {exc}")

print("imports")
from cliffguard.eval.noise_floor import difference_in_means, rotation_replication, angle_between
from cliffguard.eval.isotropy import isotropy_test
from cliffguard.eval.discriminability import (
    d_prime, d_prime_with_ci, held_out_d_prime, gaussianity_gap, implied_eta,
)
from cliffguard.eval.composition import d_prime_at_bits, collapse_bits_threshold_closed_form
from cliffguard.eval.noise_spectrum import (
    EtaMeasurement, fit_eta_vs_bits_report, projected_perturbation_variance,
)
from cliffguard.eval.storage import new_run, record_corpus, record_environment
import scripts.run_local_ladder as ladder
import scripts.run_behavioural_ladder as behav
import scripts.run_sector_ladder as sector
import scripts.classify_completions_judge as judge
print("  ok    every cliffguard module and all four runner scripts imported")

print("signatures")
for mod, names in [
    (ladder, ("rtn_quantize_dequantize", "rtn_bits_per_parameter", "load_rtn_model",
              "load_deployed_model", "deployed_quantization_config",
              "release_host_memory", "main")),
    (behav, ("classify", "has_refusal_marker", "generate_batched", "score_nll", "main")),
    (sector, ("extract_gold", "extract_predicted", "is_correct", "reusable_prefix", "main")),
    (judge, ("judge_batch", "MARKER_VARIANTS", "main")),
]:
    for n in names:
        assert hasattr(mod, n), f"{mod.__name__}.{n} missing"
print("  ok    all runner entry points present")

def _resumability():
    """The two mechanisms that decide whether an interrupted run resumes.

    Both were added after a session died at scheme seven of eight, and both fail
    silently if they regress: the ladder would simply regenerate everything and
    the notebook would look slow rather than broken. An hour of GPU time is too
    expensive to discover that from the timings.
    """
    import json as _json
    import pathlib as _pathlib
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        cache = _pathlib.Path(tmp)
        (cache / "gsm8k_FP16_n32_t192.json").write_text(
            _json.dumps([f"a{i}" for i in range(32)]), encoding="utf-8")
        # Aligned batches: reuse. Partial final batch: refuse rather than
        # silently pair differently-batched completions.
        assert sector.reusable_prefix(cache, "FP16", 16, 192, 4) == [f"a{i}" for i in range(16)]
        assert sector.reusable_prefix(cache, "FP16", 18, 192, 4) is None
    ladder.release_host_memory("preflight")
check("cache reuse and host-memory release", _resumability)

def _deployed_axis():
    """A deployed checkpoint must be gradable and must not be a rung.

    Both halves were broken at once: the judge rebuilt its scheme list from
    `bits`, which is empty for a --deployed run, so it graded nothing; and
    bits_of accepted any NAME_<digits>B, so AWQ_4B was placed at 4.5 bits and
    entered the drift fit.
    """
    import math

    from scripts.build_paper_data import bits_of
    from scripts.reanalyse_runs import ordered_schemes
    assert math.isnan(bits_of("AWQ_4B")) and bits_of("RTN_4B") == 4.5
    assert ordered_schemes({"FP16": [], "AWQ_4B": []}) == ["FP16", "AWQ_4B"]
check("deployed schemes: graded, and off the RTN axis", _deployed_axis)

rng = np.random.default_rng(0)
D, N = 64, 40
h0 = rng.normal(size=(N, D)) + np.eye(1, D, 0)[0] * 1.5
l0 = rng.normal(size=(N, D))
h1 = h0 + rng.normal(scale=0.05, size=h0.shape)
l1 = l0 + rng.normal(scale=0.05, size=l0.shape)

check("rotation_replication", lambda: rotation_replication(h0, l0, h1, l1, n_splits=5, seed=0).summary())
check("isotropy_test", lambda: isotropy_test(difference_in_means(h0, l0),
                                             difference_in_means(h1, l1), n_null=20, seed=0).summary())
check("held_out_d_prime", lambda: held_out_d_prime(h0, l0, n_splits=5, fires_high=True, seed=0))
check("d_prime_with_ci", lambda: d_prime_with_ci(rng.normal(1, 1, 200), rng.normal(0, 1, 200),
                                                 fires_high=True, n_bootstrap=50, seed=0).summary())
check("angle_between / gaussianity_gap / implied_eta",
      lambda: (angle_between(h0[0], h0[1]),
               gaussianity_gap(rng.normal(1, 1, 300), rng.normal(0, 1, 300), fires_high=True),
               implied_eta(1.0, 0.8)))
check("d_prime_at_bits / collapse_bits_threshold_closed_form",
      lambda: (d_prime_at_bits(4.0, 2.0, 0.3), collapse_bits_threshold_closed_form(2.0, 0.05, 0.3)))

def _rtn():
    w = torch.randn(64, 200, dtype=torch.float16)          # 200 % 64 != 0, exercises the tail
    err = {b: float((ladder.rtn_quantize_dequantize(w, b, 64).float() - w.float()).abs().mean())
           for b in (8, 4, 2)}
    assert err[8] < err[4] < err[2], f"RTN error not monotone in bits: {err}"
    assert ladder.rtn_bits_per_parameter(4, 64) == 4.5
check("rtn quantizer: error monotone in bits, exact bit accounting", _rtn)

def _classify():
    """Degeneracy must gate first, or token salad scores as compliance."""
    assert behav.classify("brand diidi _Nothing WL ToolsICS", 10.0, 4.4) == "degenerate"
    assert behav.classify("I cannot help with that.", 1.5, 4.4) == "refusal"
    assert behav.classify("Sure, here are the steps.", 1.5, 4.4) == "compliance"
check("three-way classifier incl. degeneracy gate", _classify)

def _gsm8k():
    assert sector.extract_gold("blah\n#### 42") == 42.0
    assert sector.extract_predicted("3 x 4 = 12, so 12 - 5 = 7.") == 7.0
    assert sector.is_correct("The answer is 42.", 42.0)
    assert not sector.is_correct("I cannot solve this.", 42.0)
check("GSM8K gold parsing and scoring", _gsm8k)

def _fit():
    m = {q: EtaMeasurement(bits_per_param_wholefile=b + .2, bits_per_param_payload=b,
                           eta=0.3 * 4.0 ** (4.0 - b))
         for q, b in {"a": 8.5, "b": 6.6, "c": 5.7, "d": 4.8, "e": 3.9}.items()}
    assert abs(fit_eta_vs_bits_report(m).exponent - 4.0) < 0.01
check("eta fit recovers a planted exponent", _fit)

def _projected():
    ones = np.ones((8, 16))
    r = rng.normal(size=8)
    assert projected_perturbation_variance(ones, ones, r) == 0.0
check("projected_perturbation_variance", _projected)

def _storage():
    r = new_run("preflight", model_id="none")
    record_environment(r)
    record_corpus(r, "x", ["a", "b"])
    r.save_array("directions", "p", np.ones(4))
    r.save_json("p", {"ok": True})
    r.write_manifest()
    import shutil
    shutil.rmtree(r.path)
check("storage.new_run / record_* / save_* / write_manifest", _storage)

def _sign():
    """r = mean(pos) - mean(neg) makes pos score HIGH; every readout is fires_high=True."""
    r = difference_in_means(h0, l0)
    r = r / np.linalg.norm(r)
    mh = (h0 @ r) / np.linalg.norm(h0, axis=1)
    ml = (l0 @ r) / np.linalg.norm(l0, axis=1)
    assert mh.mean() > ml.mean() and d_prime(mh, ml, fires_high=True) > 0
check("sign convention", _sign)

print()
if failures:
    raise SystemExit("PREFLIGHT FAILED:\n  - " + "\n  - ".join(failures))
print("PREFLIGHT OK — every module and entry point this notebook uses works.")


## Configuration and the driver

`N_GSM8K = 496` is 2.5x the paper's 200 and about a third of the full test
split. The full split costs roughly 2.3 h per model on a T4 and seven hours for
three, which does not fit a session; 496 buys the power the contested 4.5-bit
comparison needs without spending the whole run on it. The cell below says why
496 rather than 500.

Caches are written straight to Drive, so a scheme is durable the moment it
finishes rather than when the whole arm does. The helper functions are shared
with `colab_run.ipynb` so both notebooks use one harness.


In [ ]:
MODELS = [
    ("qwen15b", "Qwen/Qwen2.5-1.5B-Instruct"),
    ("qwen3b",  "Qwen/Qwen2.5-3B-Instruct"),
    ("phi35",   "microsoft/Phi-3.5-mini-instruct"),
]
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_4BIT  = VRAM_GB < 20           # a T4 cannot hold 7B in fp16

N_PROMPTS    = 250        # per source class -> 500 prompts, matching the paper
# The full 1319-question split is not affordable here. The 3B ladder measured
# ~1050 s per scheme at 1319, so one model is 2.3 h and three are seven, which
# does not fit a session. What this arm has to settle is one contested
# comparison -- Qwen2.5-3B, FP16 against 4.5 bits -- which at n=200 gave 23
# questions lost against 9 gained, p=0.020 unadjusted and 0.100 corrected. That
# needs power, not the whole split: discordant counts scale with n, so 2.5x the
# questions turns 23-against-9 into roughly 58-against-23 and the exact McNemar
# p falls by orders of magnitude.
#
# 496 rather than 500 because generation is batched at 16 and the previous
# session already produced 1319 completions for six schemes. A whole number of
# batches makes those reusable EXACTLY: the first 496 items sit in identical
# batches under both totals, so they are the same completions rather than
# approximately the same ones, and the earlier hour is not thrown away.
N_GSM8K      = 496
BITS         = ["8", "7", "6", "5", "4", "3", "2"]
LONG_BITS    = ["5", "4"] # rungs to re-run at 256 tokens, plus FP16
LONG_TOKENS  = 256
SEED         = 0
RUN_AWQ_GPTQ = True       # set False if the install fails and you want the rest

# Caches go straight to Drive when Drive is mounted, rather than to local disk
# that is mirrored afterwards. The mirror only ran between arms, so the 3B GSM8K
# ladder -- 2.3 hours of it -- was written nowhere durable until the whole model
# finished, and a disconnect at hour two lost all of it. Written here, every
# scheme is safe the moment it completes.
CACHE_ROOT = (DRIVE_ROOT / "artifacts") if DRIVE_ROOT.exists() else pathlib.Path("artifacts")
BEHAV_CACHE  = str(CACHE_ROOT / "behavioural_cache")
SECTOR_CACHE = str(CACHE_ROOT / "sector_cache")

RESULTS = {}
print(f"models   : {[m for _, m in MODELS]}")
print(f"GSM8K    : {N_GSM8K} questions (the paper used 200)")
print(f"long gen : {LONG_TOKENS} tokens at FP16 + {LONG_BITS}")
print(f"caches   : {CACHE_ROOT}"
      + ("" if DRIVE_ROOT.exists() else "   (LOCAL -- a disconnect loses them)"))

def run_step(label, script, args, timeout=10800):
    """Stream one script invocation; keep its tail and exit status.

    The timeout is enforced by a watchdog rather than by `proc.wait(timeout=...)`.
    Reading the child's stdout to EOF blocks for as long as the child lives, so
    the wait call was only ever reached after the child had already exited and
    the timeout could not fire. A step that hangs would have held the notebook
    indefinitely, which is indistinguishable from a step that is merely slow.

    A watchdog kill is recorded separately because on Linux it produces exit
    code -9 -- the same code the OOM killer produces -- and the retry logic
    below must not treat a timeout as a resumable out-of-memory event.
    """
    import threading

    cmd = [sys.executable, f"scripts/{script}"] + args
    print(f"\n$ {' '.join(cmd)}", flush=True)
    started = time.time()
    lines = []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    timed_out = []
    def _kill():
        timed_out.append(True)
        print(f"\n[{label}] no exit after {timeout / 3600:.1f} h; killing", flush=True)
        proc.kill()
    watchdog = threading.Timer(timeout, _kill)
    watchdog.daemon = True
    watchdog.start()
    try:
        for line in proc.stdout:
            if "Loading weights" in line or "it/s]" in line or "s/prompt" in line:
                continue                       # progress bars, not results
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait()
    except Exception as exc:
        proc.kill()
        proc.wait()
        lines.append(f"ABORTED: {type(exc).__name__}: {exc}")
    finally:
        watchdog.cancel()
    ok = proc.returncode == 0
    RESULTS[label] = {"returncode": proc.returncode, "timed_out": bool(timed_out),
                      "minutes": (time.time() - started) / 60, "tail": lines[-40:]}
    print(f"\n=== {label}: {'OK' if ok else f'FAILED rc={proc.returncode}'} "
          f"in {RESULTS[label]['minutes']:.1f} min ===", flush=True)
    return ok

def run_step_resumable(label, script, args, attempts=3, timeout=10800):
    """run_step, retried, because this failure mode is resumable.

    A long ladder dies with rc=-9: the Linux OOM killer. GPU memory is released
    correctly between schemes, but host RAM is not fully returned to the OS
    after each model load, and after six or seven load/unload cycles the process
    crosses Colab's limit. Observed on the 1.5B GSM8K ladder, which finished six
    of eight schemes and was killed at the seventh.

    Retrying works because every scheme writes its completions to a cache file
    the moment it finishes. A fresh process starts with those schemes already
    done, so it loads fewer models and accumulates less. Each attempt gets
    strictly further than the last, and the work converges instead of restarting.

    Only an OOM is retried. A bad flag or a missing checkpoint fails identically
    on every attempt, and pretending otherwise would turn one clear error into
    three. A watchdog timeout also reports -9 and is excluded by the flag
    run_step sets, since a step that ran out of wall clock will do so again.
    """
    tag = label
    for attempt in range(1, attempts + 1):
        # The first attempt keeps the plain label, so a step that succeeds
        # immediately reads the same as one that never needed retrying.
        tag = label if attempt == 1 else f"{label}-retry{attempt}"
        if attempt > 1:
            print(f"\n[retry {attempt}/{attempts}] {label}: resuming from cache",
                  flush=True)
        if run_step(tag, script, args, timeout=timeout):
            RESULTS[label] = RESULTS[tag]          # the label the export reads
            return True
        result = RESULTS[tag]
        if result["returncode"] != -9 or result["timed_out"]:
            reason = "timed out" if result["timed_out"] else f"rc={result['returncode']}"
            print(f"[{label}] {reason} is not a resumable OOM; not retrying", flush=True)
            RESULTS[label] = result
            return False
    print(f"[{label}] still failing after {attempts} attempts", flush=True)
    # The last tag, not a reconstructed one: with attempts=1 the only tag is the
    # bare label and "-retry1" was never written, so rebuilding the name here
    # would raise a KeyError on top of the failure it is trying to report.
    RESULTS[label] = RESULTS[tag]
    return False

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # Host memory is the one that kills this notebook. GPU allocation is
    # released between schemes; resident host memory is what ratchets up across
    # model loads until the OOM killer fires, so it is the number to watch.
    host = ""
    try:
        for line in pathlib.Path("/proc/meminfo").read_text().splitlines():
            if line.startswith("MemAvailable:"):
                host = f", host available {float(line.split()[1]) / 1e6:.1f} GB"
    except OSError:
        pass
    print(f"[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated{host}")

def checkpoint_to_drive():
    """Mirror completed run directories to Drive.

    Caches already live on Drive (see CACHE_ROOT), so only the run directories
    need copying: `new_run` writes them under a relative artifacts/runs path and
    Colab wipes local disk on disconnect.
    """
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ("runs", "behavioural_cache", "sector_cache"):
        src = pathlib.Path("artifacts") / name
        if src.exists():          # the caches are here only when Drive was absent
            shutil.copytree(src, DRIVE_ROOT / "artifacts" / name, dirs_exist_ok=True)
    print(f"[drive] mirrored artifacts/ to {DRIVE_ROOT}")

def restore_from_drive():
    """Bring back run directories from a previous session before anything runs."""
    if not DRIVE_ROOT.exists():
        return
    import shutil
    src = DRIVE_ROOT / "artifacts" / "runs"
    if src.exists():
        shutil.copytree(src, pathlib.Path("artifacts") / "runs", dirs_exist_ok=True)
        print("[drive] restored artifacts/runs")

restore_from_drive()

def latest_run(pattern):
    hits = sorted(pathlib.Path("artifacts/runs").glob(pattern))
    return hits[-1] if hits else None


## 3 — Corpus

`data/` is gitignored, so the clone has none. Built here with the repo's own
downloader, from `Anthropic/hh-rlhf` (MIT), and cached to Drive so a reconnect
does not repeat it.

**Known defect, stated up front.** These class labels come from whether hh-rlhf's
*rejected response* looks like a refusal — a property of the response, not the
prompt. They agree with a model's own behaviour about 52 % of the time, i.e.
chance. Nothing downstream uses them as harmfulness labels; the behavioural arms
derive labels from each model's own completions. The corpus is used only as a
**prompt source**.


In [ ]:
FOLD_A = pathlib.Path("data/folds/fold_a")
NEEDED = ["anthropic_hh_refused.jsonl", "anthropic_hh_benign.jsonl"]
DRIVE_FOLD = DRIVE_ROOT / "fold_a"

def have_corpus():
    return all((FOLD_A / f).exists() for f in NEEDED)

if not have_corpus() and DRIVE_FOLD.exists():
    import shutil
    FOLD_A.mkdir(parents=True, exist_ok=True)
    for f in DRIVE_FOLD.glob("*.jsonl"):
        shutil.copy(f, FOLD_A / f.name)
    print("[corpus] restored from Drive")

if not have_corpus():
    print("[corpus] building ...")
    proc = subprocess.run([sys.executable, "scripts/download_fold_a.py", "--download"],
                          capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if not have_corpus():
        print(proc.stderr[-2000:])
        raise SystemExit("corpus build failed; refusing to substitute synthetic prompts")
    if DRIVE_ROOT.exists():
        import shutil
        DRIVE_FOLD.mkdir(parents=True, exist_ok=True)
        for f in FOLD_A.glob("*.jsonl"):
            shutil.copy(f, DRIVE_FOLD / f.name)
        print("[corpus] cached to Drive")

for f in NEEDED:
    n = sum(1 for ln in (FOLD_A / f).read_text(encoding="utf-8").splitlines() if ln.strip())
    print(f"  {f}: {n} prompts")


## Step 1 — Regrade Qwen2.5-1.5B with the 7B judge

The paper excludes Qwen2.5-1.5B from the refusal arm because its original run was
graded by a 1.5B self-judge that returned REFUSE for **100%** of full-precision
completions, including plainly helpful answers. A grader with no discriminative
power cannot support a result, so the exclusion was correct — but a reviewer will
reasonably ask why the stored completions were not simply regraded with the same
7B judge the other two models used, especially since the paper claims regrading
is possible without regeneration.

The behavioural run is regenerated rather than reused because the original
predates `completion_nll.json`, and the composite degeneracy gate needs that
full-precision reference. Regenerating is also the only way to get all seven
rungs on the same footing as the other two models.

In [ ]:
free_vram()
tag, model = MODELS[0]
run_step_resumable(f"behavioural-{tag}", "run_behavioural_ladder.py",
         ["--model", model, "--n", str(N_PROMPTS), "--bits", *BITS,
          "--seed", str(SEED), "--cache", BEHAV_CACHE,
          "--label", f"r2-behavioural-{tag}"])
checkpoint_to_drive()


In [ ]:
free_vram()
run_dir = latest_run(f"*r2-behavioural-{MODELS[0][0]}")
if run_dir:
    args = [str(run_dir), "--judge-model", JUDGE_MODEL]
    if JUDGE_4BIT:
        args.append("--judge-4bit")
    run_step_resumable(f"judge-{MODELS[0][0]}", "classify_completions_judge.py", args)
else:
    print("no run directory; step 1 generation must have failed")
checkpoint_to_drive()


## Step 2 - GSM8K at 496 questions

The paper reports GSM8K on 200 questions and states outright that this lacks the
power to resolve whether Qwen2.5-3B's 4.5-bit accuracy drop is real: the paired
exact McNemar test gives p=0.020 unadjusted, 0.100 corrected within the model,
and 0.321 across every cell. The point estimate falls from 18.5% to 11.5% with 23
questions lost against 9 gained, which is suggestive and nothing more.

At 2.5x the questions the same effect gives roughly 58 lost against 23 gained,
where the exact test is decisive either way. Both answers are worth having, and
the second is worth having honestly.

Qwen2.5-3B runs first because it carries the contested comparison. If the
session ends early, the question this arm exists to answer is the one that got
answered.


In [ ]:
# Qwen2.5-3B first: it carries the contested comparison, and the other two are
# context for it. Ordering by importance rather than by model size means an
# early disconnect costs the least informative arm instead of the decisive one.
for tag in ("qwen3b", "qwen15b", "phi35"):
    model = dict(MODELS)[tag]
    free_vram()
    run_step_resumable(f"gsm8k-{tag}", "run_sector_ladder.py",
             ["--model", model, "--n", str(N_GSM8K), "--bits", *BITS,
              "--cache", SECTOR_CACHE, "--label", f"r2-gsm8k-{tag}"])
    checkpoint_to_drive()


## Step 3 — 256-token generations

48 new tokens is enough for a refusal but often not for substantive compliance,
so the design is biased toward observing refusal intact. Reading the completions
already showed that the newly-refusing ones are not truncated stubs — mean 235
characters against 246 at full precision — but that is an argument from the data
we have, not a test.

This re-runs FP16 and the two rungs carrying the result at 256 tokens on the two
models in the refusal arm. If the increase survives, the truncation objection is
answered with a measurement instead of a rebuttal.

In [ ]:
for tag, model in MODELS[1:]:          # the two models in the refusal arm
    free_vram()
    run_step_resumable(f"long-{tag}", "run_behavioural_ladder.py",
             ["--model", model, "--n", str(N_PROMPTS), "--bits", *LONG_BITS,
              "--max-new-tokens", str(LONG_TOKENS), "--seed", str(SEED),
              "--cache", BEHAV_CACHE, "--label", f"r2-long{LONG_TOKENS}-{tag}"])
    checkpoint_to_drive()


In [ ]:
for tag, _ in MODELS[1:]:
    free_vram()
    run_dir = latest_run(f"*r2-long{LONG_TOKENS}-{tag}")
    if run_dir:
        args = [str(run_dir), "--judge-model", JUDGE_MODEL]
        if JUDGE_4BIT:
            args.append("--judge-4bit")
        run_step_resumable(f"judge-long-{tag}", "classify_completions_judge.py", args)
    else:
        print(f"no r2-long{LONG_TOKENS}-{tag} run directory; step 3 must have failed")
checkpoint_to_drive()


## Step 4 — AWQ and GPTQ (optional, install-fragile)

RTN varies only bit-width, which is exactly why the paper uses it: it is the
clean instrument, and a mixed-type format would vary block structure and
per-tensor type assignment at the same time, leaving any threshold
uninterpretable. The cost is external validity, since nobody deploys RTN.

Pre-quantized AWQ and GPTQ checkpoints test whether the direction survives a
quantizer people actually ship. This is the step most likely to fail on Colab —
`autoawq` and `auto-gptq` pull heavy dependencies and are sensitive to the CUDA
build — so it is wrapped to record a failure and continue.

Note these are fixed 4-bit checkpoints, not a ladder: the comparison is FP16
against one deployed quantizer, not a dose-response curve.

In [ ]:
if RUN_AWQ_GPTQ:
    installed = subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install", "autoawq", "optimum",
         "gptqmodel"], capture_output=True).returncode == 0
    print("AWQ/GPTQ install:", "ok" if installed else "FAILED - skipping step 4")

    if installed:
        # LABEL=REPO. The label names the scheme everywhere downstream; bits_of()
        # already reads "AWQ_4B" as 4.5 stored bits, so these land on the same
        # axis as the RTN rungs without any analysis change.
        #
        # --model stays the FP16 base and --bits is empty: these checkpoints are
        # ALREADY quantized, so they are extra schemes paired against the same
        # FP16 baseline, not models to apply RTN to. Passing them to --model with
        # --bits would quantize them twice.
        DEPLOYED = [
            ("qwen3b", "Qwen/Qwen2.5-3B-Instruct", [
                "AWQ_4B=Qwen/Qwen2.5-3B-Instruct-AWQ",
                "GPTQ_4B=Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
            ]),
        ]
        for tag, base, specs in DEPLOYED:
            free_vram()
            run_step_resumable(f"deployed-{tag}", "run_behavioural_ladder.py",
                     ["--model", base, "--n", str(N_PROMPTS), "--bits",
                      "--deployed", *specs, "--seed", str(SEED),
                      "--cache", BEHAV_CACHE, "--label", f"r2-deployed-{tag}"])
            checkpoint_to_drive()

        # Grade them with the same judge as everything else, or the comparison
        # is against a different instrument rather than a different quantizer.
        free_vram()
        for tag, _, _ in DEPLOYED:
            run_dir = latest_run(f"*r2-deployed-{tag}")
            if run_dir:
                args = [str(run_dir), "--judge-model", JUDGE_MODEL]
                if JUDGE_4BIT:
                    args.append("--judge-4bit")
                run_step_resumable(f"judge-deployed-{tag}", "classify_completions_judge.py", args)
        checkpoint_to_drive()
else:
    print("step 4 skipped by configuration")

## Export

One zip with every `r2-*` run directory, stored with repository-relative paths.
The first run's directories are left alone, so both sets sit side by side.

Download it, unzip at the repository root, and run the two commands the cell
prints. Both take `--include '*r2-*'`: round one's runs are still present, and
the analysis scripts stop rather than silently choose between two runs of the
same model.


In [ ]:
import zipfile

runs = sorted(pathlib.Path("artifacts/runs").glob("*r2-*"))
print("run directories from this notebook:")
for r in runs:
    print("  ", r.name)

# Two directories with the same label mean the notebook ran twice. Both are
# valid, but every analysis downstream refuses to choose between two runs of one
# model, so say it here rather than letting it surface as a SystemExit later.
by_label = {}
for r in runs:
    by_label.setdefault(r.name.split("_", 2)[-1], []).append(r.name)
duplicates = {k: v for k, v in by_label.items() if len(v) > 1}
if duplicates:
    print("\nNOTE: repeated labels. Keep the newest of each and delete the rest,")
    print("or the analysis scripts will stop rather than pick one:")
    for label, names in duplicates.items():
        print(f"  {label}: {names}  -> keep {sorted(names)[-1]}")

# Canonical labels only: a retried step also appears under a -retry tag, and
# listing both would report one failure twice.
steps = {k: v for k, v in RESULTS.items() if "-retry" not in k}
failed = [k for k, v in steps.items() if v.get("returncode") != 0]
retried = sorted({k.split("-retry")[0] for k in RESULTS if "-retry" in k})

summary = {"steps": RESULTS, "runs": [r.name for r in runs],
           "failed": failed, "retried": retried,
           "n_gsm8k": N_GSM8K, "n_prompts": N_PROMPTS}
status = pathlib.Path("artifacts/runs/ROUND2_STATUS.json")
status.write_text(json.dumps(summary, indent=2), encoding="utf-8")

stamp = time.strftime("%Y%m%d-%H%M%S")
archive = pathlib.Path((f"/content/cliffguard_round2_{stamp}.zip" if IN_COLAB
                        else f"cliffguard_round2_{stamp}.zip"))
# Paths are stored relative to the repository root, so "unzip at the repository
# root" is literally what to do. Only r2-* directories go in: the first run's
# are already on this machine and would double the download.
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(status, status.as_posix())
    for run in runs:
        for path in sorted(run.rglob("*")):
            if path.is_file():
                zf.write(path, path.as_posix())
print(f"\nwrote {archive}  ({archive.stat().st_size / 1e6:.1f} MB)")

print("\nSTEPS:")
for k, v in steps.items():
    mark = "ok  " if v.get("returncode") == 0 else f"FAIL({v.get('returncode')})"
    print(f"  {mark} {k:26s} {v.get('minutes', 0):6.1f} min")
print("FAILED STEPS:", failed if failed else "none")
if retried:
    print("resumed after an OOM:", retried)

print("""
Next, locally, from the repository root:

  unzip cliffguard_round2_*.zip
  python scripts/analyse_round2.py --gsm8k <gsm8k test.jsonl>

That script, not review_reanalysis.py. review_reanalysis fits a ladder: it wants
one behavioural run per model with enough rungs to regress on, and it stops when
two runs describe the same model. Round two writes two runs of Qwen2.5-3B, a
deployed run with no RTN rungs at all, and two ladders three rungs long, so it
would refuse before it started -- correctly. analyse_round2.py runs the paired
tests these four arms actually pose, keyed by run label, and it reads BOTH rounds
on purpose: the 256-token question is only answerable against round one's
48-token run on the same prompts.

The paper's published numbers are unaffected and still reproduce with

  python scripts/review_reanalysis.py --exclude '*r2-*' --gsm8k <gsm8k test.jsonl>
""")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as exc:
        print("download it from the file browser instead:", exc)
